In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [5]:
df = pd.read_csv("../data/processed/olist_orders_abt.csv")

print("Shape:", df.shape)
df.head()


Shape: (99441, 29)


,order_id,customer_id,customer_unique_id,customer_city,customer_state,order_status,order_year,order_month,order_day,order_day_of_week,...,total_payment_value,max_payment_installments,payment_types_count,dominant_payment_type,total_items,total_price,total_freight,unique_products,unique_sellers,main_product_category
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,delivered,2017,10,2,0,...,38.71,1.0,2.0,voucher,1.0,29.99,8.72,1.0,1.0,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,barreiras,BA,delivered,2018,7,24,1,...,141.46,1.0,1.0,boleto,1.0,118.70,22.76,1.0,1.0,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,delivered,2018,8,8,2,...,179.12,3.0,1.0,credit_card,1.0,159.90,19.22,1.0,1.0,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,delivered,2017,11,18,5,...,72.20,1.0,1.0,credit_card,1.0,45.00,27.20,1.0,1.0,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,delivered,2018,2,13,1,...,28.62,1.0,1.0,credit_card,1.0,19.90,8.72,1.0,1.0,stationery


In [6]:
df.columns

Index(['order_id', 'customer_id', 'customer_unique_id', 'customer_city',
       'customer_state', 'order_status', 'order_year', 'order_month',
       'order_day', 'order_day_of_week', 'order_hour', 'delivery_days',
       'estimated_delivery_days', 'delivery_delay_days', 'is_late_delivery',
       'review_score', 'is_low_review', 'review_comment_count',
       'has_review_comment', 'total_payment_value', 'max_payment_installments',
       'payment_types_count', 'dominant_payment_type', 'total_items',
       'total_price', 'total_freight', 'unique_products', 'unique_sellers',
       'main_product_category'],
      dtype='str')

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   order_id                  99441 non-null  str    
 1   customer_id               99441 non-null  str    
 2   customer_unique_id        99441 non-null  str    
 3   customer_city             99441 non-null  str    
 4   customer_state            99441 non-null  str    
 5   order_status              99441 non-null  str    
 6   order_year                99441 non-null  int64  
 7   order_month               99441 non-null  int64  
 8   order_day                 99441 non-null  int64  
 9   order_day_of_week         99441 non-null  int64  
 10  order_hour                99441 non-null  int64  
 11  delivery_days             96476 non-null  float64
 12  estimated_delivery_days   99441 non-null  int64  
 13  delivery_delay_days       96476 non-null  float64
 14  is_late_delivery 

In [8]:
identifier_columns = [
    "order_id",
    "customer_id",
    "customer_unique_id"
]

leakage_columns = [
    "delivery_days",
    "delivery_delay_days",
    "review_score",
    "review_comment_count",
    "has_review_comment",
    "is_low_review"
]

columns_to_drop = identifier_columns + leakage_columns

feature_df = df.drop(columns=columns_to_drop)

print("Original shape:", df.shape)
print("New shape:", feature_df.shape)

Original shape: (99441, 29)
New shape: (99441, 20)


In [9]:
feature_df.columns

Index(['customer_city', 'customer_state', 'order_status', 'order_year',
       'order_month', 'order_day', 'order_day_of_week', 'order_hour',
       'estimated_delivery_days', 'is_late_delivery', 'total_payment_value',
       'max_payment_installments', 'payment_types_count',
       'dominant_payment_type', 'total_items', 'total_price', 'total_freight',
       'unique_products', 'unique_sellers', 'main_product_category'],
      dtype='str')

In [10]:
feature_df["average_item_price"] = (
    feature_df["total_price"] /
    feature_df["total_items"].replace(0, np.nan)
)

In [11]:
feature_df["freight_ratio"] = (
    feature_df["total_freight"] /
    feature_df["total_price"].replace(0, np.nan)
)

In [12]:
feature_df["items_per_seller"] = (
    feature_df["total_items"] /
    feature_df["unique_sellers"].replace(0, np.nan)
)

In [13]:
feature_df["seller_diversity"] = (
    feature_df["unique_sellers"] /
    feature_df["total_items"].replace(0, np.nan)
)

In [14]:
feature_df[
    [
        "average_item_price",
        "freight_ratio",
        "items_per_seller",
        "seller_diversity"
    ]
].head()

,average_item_price,freight_ratio,items_per_seller,seller_diversity
0,29.99,0.290764,1.0,1.0
1,118.70,0.191744,1.0,1.0
2,159.90,0.120200,1.0,1.0
3,45.00,0.604444,1.0,1.0
4,19.90,0.438191,1.0,1.0


In [15]:
feature_df["is_weekend"] = (
    feature_df["order_day_of_week"] >= 5
).astype(int)

In [16]:
feature_df["is_business_hour"] = (
    feature_df["order_hour"].between(9, 18)
).astype(int)

In [17]:
feature_df["is_year_end"] = (
    feature_df["order_month"].isin([11, 12])
).astype(int)

In [18]:
feature_df[
    [
        "order_month",
        "order_day_of_week",
        "order_hour",
        "is_weekend",
        "is_business_hour",
        "is_year_end"
    ]
].head(10)

,order_month,order_day_of_week,order_hour,is_weekend,is_business_hour,is_year_end
0,10,0,10,0,1,0
1,7,1,20,0,0,0
2,8,2,8,0,0,0
3,11,5,19,1,0,1
4,2,1,21,0,0,0
5,7,6,21,1,0,0
6,4,1,12,0,1,0
7,5,1,13,0,1,0
8,1,0,18,0,1,0
9,7,5,11,1,1,0


In [19]:
feature_df["month_sin"] = np.sin(
    2 * np.pi * feature_df["order_month"] / 12
)

feature_df["month_cos"] = np.cos(
    2 * np.pi * feature_df["order_month"] / 12
)

In [20]:
feature_df["hour_sin"] = np.sin(
    2 * np.pi * feature_df["order_hour"] / 24
)

feature_df["hour_cos"] = np.cos(
    2 * np.pi * feature_df["order_hour"] / 24
)

In [21]:
feature_df[
    [
        "order_month",
        "month_sin",
        "month_cos",
        "order_hour",
        "hour_sin",
        "hour_cos"
    ]
].head(10)

,order_month,month_sin,month_cos,order_hour,hour_sin,hour_cos
0,10,-0.866025,0.500000,10,5.000000e-01,-8.660254e-01
1,7,-0.500000,-0.866025,20,-8.660254e-01,5.000000e-01
2,8,-0.866025,-0.500000,8,8.660254e-01,-5.000000e-01
3,11,-0.500000,0.866025,19,-9.659258e-01,2.588190e-01
4,2,0.866025,0.500000,21,-7.071068e-01,7.071068e-01
5,7,-0.500000,-0.866025,21,-7.071068e-01,7.071068e-01
6,4,0.866025,-0.500000,12,1.224647e-16,-1.000000e+00
7,5,0.500000,-0.866025,13,-2.588190e-01,-9.659258e-01
8,1,0.500000,0.866025,18,-1.000000e+00,-1.836970e-16
9,7,-0.500000,-0.866025,11,2.588190e-01,-9.659258e-01


In [22]:
feature_df["log_total_price"] = np.log1p(
    feature_df["total_price"].clip(lower=0)
)

In [23]:
feature_df["log_total_freight"] = np.log1p(
    feature_df["total_freight"].clip(lower=0)
)

In [24]:
feature_df[
    [
        "total_price",
        "log_total_price",
        "total_freight",
        "log_total_freight"
    ]
].head(10)

,total_price,log_total_price,total_freight,log_total_freight
0,29.99,3.433665,8.72,2.274186
1,118.70,4.784989,22.76,3.168003
2,159.90,5.080783,19.22,3.006672
3,45.00,3.828641,27.20,3.339322
4,19.90,3.039749,8.72,2.274186
5,147.90,5.003275,27.36,3.344980
6,49.90,3.929863,16.05,2.836150
7,59.99,4.110710,15.17,2.783158
8,19.90,3.039749,16.05,2.836150
9,149.99,5.017214,19.77,3.033510


In [25]:
feature_df["price_band"] = pd.cut(
    feature_df["total_price"],
    bins=[-np.inf, 100, 500, 1000, 5000, np.inf],
    labels=[
        "very_low",
        "low",
        "medium",
        "high",
        "very_high"
    ]
)

In [26]:
feature_df[
    [
        "total_price",
        "price_band"
    ]
].head(20)

,total_price,price_band
0,29.99,very_low
1,118.70,low
2,159.90,low
3,45.00,very_low
4,19.90,very_low
5,147.90,low
6,49.90,very_low
7,59.99,very_low
8,19.90,very_low
9,149.99,low


In [27]:
feature_df["items_x_price"] = (
    feature_df["total_items"] *
    feature_df["total_price"]
)

In [28]:
feature_df[
    [
        "total_price",
        "log_total_price",
        "total_freight",
        "log_total_freight",
        "price_band",
        "total_items",
        "items_x_price"
    ]
].head(10)

,total_price,log_total_price,total_freight,log_total_freight,price_band,total_items,items_x_price
0,29.99,3.433665,8.72,2.274186,very_low,1.0,29.99
1,118.70,4.784989,22.76,3.168003,low,1.0,118.70
2,159.90,5.080783,19.22,3.006672,low,1.0,159.90
3,45.00,3.828641,27.20,3.339322,very_low,1.0,45.00
4,19.90,3.039749,8.72,2.274186,very_low,1.0,19.90
5,147.90,5.003275,27.36,3.344980,low,1.0,147.90
6,49.90,3.929863,16.05,2.836150,very_low,1.0,49.90
7,59.99,4.110710,15.17,2.783158,very_low,1.0,59.99
8,19.90,3.039749,16.05,2.836150,very_low,1.0,19.90
9,149.99,5.017214,19.77,3.033510,low,1.0,149.99


In [29]:
engineered_columns = [
    "average_item_price",
    "freight_ratio",
    "items_per_seller",
    "seller_diversity",
    "is_weekend",
    "is_business_hour",
    "is_year_end",
    "month_sin",
    "month_cos",
    "hour_sin",
    "hour_cos",
    "log_total_price",
    "log_total_freight",
    "price_band",
    "items_x_price"
]

feature_df[engineered_columns].isna().sum()

average_item_price    775
freight_ratio         775
items_per_seller      775
seller_diversity      775
is_weekend              0
is_business_hour        0
is_year_end             0
month_sin               0
month_cos               0
hour_sin                0
hour_cos                0
log_total_price       775
log_total_freight     775
price_band            775
items_x_price         775
dtype: int64

In [30]:
feature_df[engineered_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
average_item_price,98666.0,125.919255,190.985636,0.850000,41.990000,7.900000e+01,1.399000e+02,6735.000000
freight_ratio,98666.0,0.308389,0.314762,0.000000,0.131864,2.243740e-01,3.801909e-01,21.447059
items_per_seller,98666.0,1.125595,0.507908,1.000000,1.000000,1.000000e+00,1.000000e+00,21.000000
seller_diversity,98666.0,0.951276,0.158556,0.047619,1.000000,1.000000e+00,1.000000e+00,1.000000
is_weekend,99441.0,0.229754,0.420677,0.000000,0.000000,0.000000e+00,0.000000e+00,1.000000
is_business_hour,99441.0,0.620167,0.485348,0.000000,0.000000,1.000000e+00,1.000000e+00,1.000000
is_year_end,99441.0,0.132923,0.339493,0.000000,0.000000,0.000000e+00,0.000000e+00,1.000000
month_sin,99441.0,0.077961,0.694474,-1.000000,-0.500000,1.224647e-16,8.660254e-01,1.000000
month_cos,99441.0,-0.117340,0.705598,-1.000000,-0.866025,-5.000000e-01,5.000000e-01,1.000000
hour_sin,99441.0,-0.331782,0.592104,-1.000000,-0.866025,-5.000000e-01,1.224647e-16,1.000000


In [31]:
feature_df[engineered_columns].head()

,average_item_price,freight_ratio,items_per_seller,seller_diversity,is_weekend,is_business_hour,is_year_end,month_sin,month_cos,hour_sin,hour_cos,log_total_price,log_total_freight,price_band,items_x_price
0,29.99,0.290764,1.0,1.0,0,1,0,-0.866025,0.500000,0.500000,-0.866025,3.433665,2.274186,very_low,29.99
1,118.70,0.191744,1.0,1.0,0,0,0,-0.500000,-0.866025,-0.866025,0.500000,4.784989,3.168003,low,118.70
2,159.90,0.120200,1.0,1.0,0,0,0,-0.866025,-0.500000,0.866025,-0.500000,5.080783,3.006672,low,159.90
3,45.00,0.604444,1.0,1.0,1,0,1,-0.500000,0.866025,-0.965926,0.258819,3.828641,3.339322,very_low,45.00
4,19.90,0.438191,1.0,1.0,0,0,0,0.866025,0.500000,-0.707107,0.707107,3.039749,2.274186,very_low,19.90


In [32]:
from sklearn.feature_selection import VarianceThreshold

In [33]:
numeric_df = feature_df.select_dtypes(include=np.number).copy()

numeric_df = numeric_df.drop(
    columns=["is_late_delivery"],
    errors="ignore"
)

print("Number of numerical features:", numeric_df.shape[1])

Number of numerical features: 28


In [34]:
selector = VarianceThreshold(threshold=0.0)

selector.fit(
    numeric_df.fillna(numeric_df.median())
)

VarianceThreshold()

In [35]:
threshold=0.0

In [36]:
selected_numeric = numeric_df.columns[
    selector.get_support()
]

print("Selected numerical features:")
print(selected_numeric.tolist())

print("\nNumber selected:", len(selected_numeric))

Selected numerical features:
['order_year', 'order_month', 'order_day', 'order_day_of_week', 'order_hour', 'estimated_delivery_days', 'total_payment_value', 'max_payment_installments', 'payment_types_count', 'total_items', 'total_price', 'total_freight', 'unique_products', 'unique_sellers', 'average_item_price', 'freight_ratio', 'items_per_seller', 'seller_diversity', 'is_weekend', 'is_business_hour', 'is_year_end', 'month_sin', 'month_cos', 'hour_sin', 'hour_cos', 'log_total_price', 'log_total_freight', 'items_x_price']

Number selected: 28


In [37]:
removed_numeric = numeric_df.columns[
    ~selector.get_support()
]

print("Features removed because of zero variance:")
print(removed_numeric.tolist())

Features removed because of zero variance:
[]


In [38]:
corr = numeric_df.corr(numeric_only=True)

corr

,order_year,order_month,order_day,order_day_of_week,order_hour,estimated_delivery_days,total_payment_value,max_payment_installments,payment_types_count,total_items,...,is_weekend,is_business_hour,is_year_end,month_sin,month_cos,hour_sin,hour_cos,log_total_price,log_total_freight,items_x_price
order_year,1.000000,-0.550059,-0.043672,-0.017917,-0.003229,-0.142717,-0.000183,-0.059538,-0.019503,0.000211,...,-0.007842,0.014861,-0.418601,0.384586,-0.239787,0.002193,-0.018217,0.002420,0.006291,-0.001697
order_month,-0.550059,1.000000,0.001343,0.020588,-0.003953,-0.094182,0.002795,0.030311,0.003532,0.000457,...,0.006949,-0.013991,0.653618,-0.779053,0.104784,0.005060,0.013300,0.007743,0.008077,0.002842
order_day,-0.043672,0.001343,1.000000,-0.024895,-0.012195,-0.030391,-0.008652,0.001596,0.002613,0.011453,...,-0.012283,0.004332,0.049311,-0.007958,0.040667,0.008465,-0.005243,-0.012869,-0.006626,0.001063
order_day_of_week,-0.017917,0.020588,-0.024895,1.000000,0.009433,0.067685,-0.002538,0.026230,-0.000564,-0.011766,...,0.768693,-0.020495,0.036749,-0.002184,0.021455,-0.027702,0.030770,-0.007037,0.001661,-0.004792
order_hour,-0.003229,-0.003953,-0.012195,0.009433,1.000000,-0.000144,0.004185,0.013405,0.005917,-0.009770,...,0.043324,-0.284269,-0.002924,0.001749,-0.002273,-0.731042,0.441757,0.009325,0.004170,-0.003033
estimated_delivery_days,-0.142717,-0.094182,-0.030391,0.067685,-0.000144,1.000000,0.094915,0.094111,0.011618,0.015000,...,0.038642,-0.004614,0.042641,0.158750,0.130545,-0.005420,0.008675,0.119597,0.369335,0.039232
total_payment_value,-0.000183,0.002795,-0.008652,-0.002538,0.004185,0.094915,1.000000,0.319090,-0.006926,0.189216,...,-0.004013,0.013845,-0.007296,-0.000972,-0.011767,-0.009448,-0.010998,0.728198,0.447608,0.662697
max_payment_installments,-0.059538,0.030311,0.001596,0.026230,0.013405,0.094111,0.319090,1.000000,-0.042044,0.067573,...,0.028445,-0.019672,0.013966,-0.022352,-0.015839,-0.015549,0.027149,0.409233,0.244892,0.146641
payment_types_count,-0.019503,0.003532,0.002613,-0.000564,0.005917,0.011618,-0.006926,-0.042044,1.000000,-0.006919,...,-0.005315,-0.005562,0.000888,-0.003713,0.002748,0.000049,0.005410,-0.008241,0.003781,-0.006702
total_items,0.000211,0.000457,0.011453,-0.011766,-0.009770,0.015000,0.189216,0.067573,-0.006919,1.000000,...,-0.014956,0.017490,0.002483,0.000320,0.005730,0.003848,-0.016226,0.179343,0.397014,0.447992


In [39]:
corr_threshold = 0.90

upper = corr.where(
    np.triu(
        np.ones(corr.shape),
        k=1
    ).astype(bool)
)

high_corr_pairs = []

for col in upper.columns:
    for row in upper.index:
        value = upper.loc[row, col]

        if pd.notna(value) and abs(value) > corr_threshold:
            high_corr_pairs.append(
                (row, col, value)
            )

high_corr_pairs

[('total_payment_value', 'total_price', np.float64(0.9959698733878193)),
 ('total_payment_value', 'average_item_price', np.float64(0.9211962096778621)),
 ('total_price', 'average_item_price', np.float64(0.9331693361267265)),
 ('total_items', 'items_per_seller', np.float64(0.9573595867372893))]

In [40]:
for feature1, feature2, correlation in high_corr_pairs:
    print(
        feature1,
        "<-->",
        feature2,
        ":",
        round(correlation, 3)
    )

total_payment_value <--> total_price : 0.996
total_payment_value <--> average_item_price : 0.921
total_price <--> average_item_price : 0.933
total_items <--> items_per_seller : 0.957


In [41]:
from sklearn.feature_selection import mutual_info_classif

In [42]:
mi_df = feature_df.select_dtypes(include=np.number).copy()

mi_df = mi_df.drop(
    columns=["is_late_delivery"],
    errors="ignore"
)

mi_df = mi_df.fillna(mi_df.median())

y = feature_df["is_late_delivery"]

In [43]:
mi_scores = mutual_info_classif(
    mi_df,
    y,
    random_state=42
)

In [44]:
mi_results = pd.DataFrame({
    "feature": mi_df.columns,
    "mutual_information": mi_scores
}).sort_values(
    "mutual_information",
    ascending=False
)

mi_results

,feature,mutual_information
13,unique_sellers,0.026495
12,unique_products,0.024199
19,is_business_hour,0.018723
8,payment_types_count,0.015779
17,seller_diversity,0.014977
1,order_month,0.013155
22,month_cos,0.011267
16,items_per_seller,0.011014
9,total_items,0.010325
21,month_sin,0.009096


In [45]:
mi_results.head(15)

,feature,mutual_information
13,unique_sellers,0.026495
12,unique_products,0.024199
19,is_business_hour,0.018723
8,payment_types_count,0.015779
17,seller_diversity,0.014977
1,order_month,0.013155
22,month_cos,0.011267
16,items_per_seller,0.011014
9,total_items,0.010325
21,month_sin,0.009096


In [46]:
# Check the final feature columns

print("Features currently in feature_df:")
print(feature_df.columns.tolist())

Features currently in feature_df:
['customer_city', 'customer_state', 'order_status', 'order_year', 'order_month', 'order_day', 'order_day_of_week', 'order_hour', 'estimated_delivery_days', 'is_late_delivery', 'total_payment_value', 'max_payment_installments', 'payment_types_count', 'dominant_payment_type', 'total_items', 'total_price', 'total_freight', 'unique_products', 'unique_sellers', 'main_product_category', 'average_item_price', 'freight_ratio', 'items_per_seller', 'seller_diversity', 'is_weekend', 'is_business_hour', 'is_year_end', 'month_sin', 'month_cos', 'hour_sin', 'hour_cos', 'log_total_price', 'log_total_freight', 'price_band', 'items_x_price']


In [47]:
# Check whether any leakage-prone columns are still present

leakage_check = [
    "delivery_days",
    "delivery_delay_days",
    "review_score",
    "review_comment_count",
    "has_review_comment",
    "is_low_review"
]

remaining_leakage = [
    col for col in leakage_check
    if col in feature_df.columns
]

print("Remaining leakage columns:", remaining_leakage)

Remaining leakage columns: []


In [48]:
candidate_features = [
    "total_price",
    "total_freight",
    "total_items",
    "unique_products",
    "unique_sellers",
    "order_month",
    "order_day_of_week",
    "order_hour",
    "average_item_price",
    "freight_ratio",
    "items_per_seller",
    "seller_diversity",
    "is_weekend",
    "is_business_hour",
    "is_year_end",
    "month_sin",
    "month_cos",
    "hour_sin",
    "hour_cos",
    "log_total_price",
    "log_total_freight",
    "items_x_price"
]

target = "is_late_delivery"

final_features = [
    col for col in candidate_features
    if col in feature_df.columns
]

final_df = feature_df[final_features + [target]].copy()

print("Final dataset shape:", final_df.shape)
print("\nFinal features:")
print(final_df.columns.tolist())

Final dataset shape: (99441, 23)

Final features:
['total_price', 'total_freight', 'total_items', 'unique_products', 'unique_sellers', 'order_month', 'order_day_of_week', 'order_hour', 'average_item_price', 'freight_ratio', 'items_per_seller', 'seller_diversity', 'is_weekend', 'is_business_hour', 'is_year_end', 'month_sin', 'month_cos', 'hour_sin', 'hour_cos', 'log_total_price', 'log_total_freight', 'items_x_price', 'is_late_delivery']


In [49]:
output_path = "../data/processed/olist_orders_feature_engineered.csv"

final_df.to_csv(output_path, index=False)

print("Saved successfully!")
print("File:", output_path)

Saved successfully!
File: ../data/processed/olist_orders_feature_engineered.csv


In [50]:
# Verify that the saved file can be loaded

check_df = pd.read_csv(output_path)

print("Saved dataset shape:", check_df.shape)
check_df.head()

Saved dataset shape: (99441, 23)


,total_price,total_freight,total_items,unique_products,unique_sellers,order_month,order_day_of_week,order_hour,average_item_price,freight_ratio,...,is_business_hour,is_year_end,month_sin,month_cos,hour_sin,hour_cos,log_total_price,log_total_freight,items_x_price,is_late_delivery
0,29.99,8.72,1.0,1.0,1.0,10,0,10,29.99,0.290764,...,1,0,-0.866025,0.500000,0.500000,-0.866025,3.433665,2.274186,29.99,0
1,118.70,22.76,1.0,1.0,1.0,7,1,20,118.70,0.191744,...,0,0,-0.500000,-0.866025,-0.866025,0.500000,4.784989,3.168003,118.70,0
2,159.90,19.22,1.0,1.0,1.0,8,2,8,159.90,0.120200,...,0,0,-0.866025,-0.500000,0.866025,-0.500000,5.080783,3.006672,159.90,0
3,45.00,27.20,1.0,1.0,1.0,11,5,19,45.00,0.604444,...,0,1,-0.500000,0.866025,-0.965926,0.258819,3.828641,3.339322,45.00,0
4,19.90,8.72,1.0,1.0,1.0,2,1,21,19.90,0.438191,...,0,0,0.866025,0.500000,-0.707107,0.707107,3.039749,2.274186,19.90,0


In [51]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [52]:
baseline_features = [
    "total_price",
    "total_freight",
    "total_items",
    "unique_products",
    "unique_sellers",
    "order_month",
    "order_day_of_week",
    "order_hour"
]

X_baseline = df[baseline_features].copy()
y = df["is_late_delivery"]

X_baseline = X_baseline.fillna(X_baseline.median())

In [53]:
engineered_features = [
    col for col in final_df.columns
    if col != "is_late_delivery"
]

X_engineered = final_df[engineered_features].copy()
X_engineered = X_engineered.fillna(X_engineered.median())

y_engineered = final_df["is_late_delivery"]

In [54]:
Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    X_baseline,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Xe_train, Xe_test, ye_train, ye_test = train_test_split(
    X_engineered,
    y_engineered,
    test_size=0.2,
    random_state=42,
    stratify=y_engineered
)

In [55]:
baseline_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

baseline_model.fit(Xb_train, yb_train)

baseline_pred = baseline_model.predict(Xb_test)

baseline_accuracy = accuracy_score(
    yb_test,
    baseline_pred
)

print("Baseline Accuracy:", baseline_accuracy)

Baseline Accuracy: 0.9134194781034742


In [56]:
engineered_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

engineered_model.fit(Xe_train, ye_train)

engineered_pred = engineered_model.predict(Xe_test)

engineered_accuracy = accuracy_score(
    ye_test,
    engineered_pred
)

print("Engineered Accuracy:", engineered_accuracy)

Engineered Accuracy: 0.9153803610035698


In [57]:
print("Baseline Accuracy   :", round(baseline_accuracy, 4))
print("Engineered Accuracy :", round(engineered_accuracy, 4))

improvement = engineered_accuracy - baseline_accuracy

print("Improvement          :", round(improvement, 4))

Baseline Accuracy   : 0.9134
Engineered Accuracy : 0.9154
Improvement          : 0.002


Conclusion:
In this lab, feature engineering and feature selection were performed on the Olist Brazilian E-Commerce dataset to improve the representation of information for machine learning. Several numerical, ratio, date/time, cyclical, logarithmic, and interaction features were created. Identifier and leakage-prone features were removed to ensure that the model uses information available at prediction time. Variance, correlation, mutual information, and Random Forest feature importance were explored for feature selection. Finally, the baseline and feature-engineered models were compared using the same experimental setup. The experiment demonstrated that good feature engineering should be guided by business meaning and that adding more features does not necessarily guarantee better model performance.